In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, mean_squared_error

raiz_projeto = str(Path().resolve().parent)
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

from src.data_loader import SEEDS_TREINO, fixar_sementes, preparar_dados_pytorch
from src.models import MLP
from src.train import treinar

os.makedirs("../models", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# 1. Nomes exatos das features na ordem usada
df_reg_cols = pd.read_csv("../data/processed/dataset_regression.csv")
features_reg = [c for c in df_reg_cols.columns if c != "term_gpa"]

df_cls_cols = pd.read_csv("../data/processed/dataset_classification.csv")
features_cls = [c for c in df_cls_cols.columns if c != "gpa_category_encoded"]

# 2. Carregamento dos DataLoaders (mesma divisão 80/20 e random_state=42)
fixar_sementes(42)
loaders_reg, scaler_X_reg, scaler_y_reg, n_features_reg = preparar_dados_pytorch(
    caminho_csv="../data/processed/dataset_regression.csv",
    target_col="term_gpa",
    tipo="reg",
    batch_size=32,
)

loaders_cls, scaler_X_cls, _, n_features_cls = preparar_dados_pytorch(
    caminho_csv="../data/processed/dataset_classification.csv",
    target_col="gpa_category_encoded",
    tipo="class",
    batch_size=32,
)

In [3]:
# 3. Configurações vencedoras do estudo de hiperparâmetros (Tarefa 3)
CFG_REG = {"in_dim": n_features_reg, "out_dim": 1, "hidden": [128, 128], "activation": "ReLU", "dropout": 0.0}
LR_REG = 1e-2

CFG_CLS = {"in_dim": n_features_cls, "out_dim": 1, "hidden": [128, 128], "activation": "ReLU", "dropout": 0.0}
LR_CLS = 1e-3
LIMIAR_CLS = 0.5

In [4]:
def treinar_melhor_semente(loaders, cfg, lr, loss_fn):
    """Treina nas sementes e retorna o modelo com menor perda na validação."""
    melhor_mod = None
    melhor_val = float("inf")
    melhor_seed = None

    for seed in SEEDS_TREINO:
        fixar_sementes(seed)
        mod = MLP(
            in_dim=cfg["in_dim"],
            out_dim=cfg["out_dim"],
            hidden=tuple(cfg["hidden"]),
            activation=cfg["activation"],
            dropout=cfg["dropout"],
        )
        opt = torch.optim.Adam(mod.parameters(), lr=lr)
        hist = treinar(
            modelo=mod,
            dl_tr=loaders["train"],
            dl_val=loaders["val"],
            loss_fn=loss_fn,
            otimizador=opt,
            n_epocas=200,
            paciencia=20,
            device=device,
        )
        if min(hist["val"]) < melhor_val:
            melhor_val = min(hist["val"])
            melhor_mod = mod
            melhor_seed = seed

    return melhor_mod, melhor_seed

In [5]:
# Treina os dois modelos finais
modelo_reg, seed_reg = treinar_melhor_semente(loaders_reg, CFG_REG, LR_REG, nn.MSELoss())
modelo_cls, seed_cls = treinar_melhor_semente(loaders_cls, CFG_CLS, LR_CLS, nn.BCEWithLogitsLoss())

# Extrai o conjunto de teste inteiro para guardar referência de fidelidade antes de salvar
X_test_reg, y_test_reg = next(iter(torch.utils.data.DataLoader(loaders_reg["test"].dataset, batch_size=len(loaders_reg["test"].dataset))))
X_test_cls, y_test_cls = next(iter(torch.utils.data.DataLoader(loaders_cls["test"].dataset, batch_size=len(loaders_cls["test"].dataset))))

modelo_reg.cpu().eval()
modelo_cls.cpu().eval()

with torch.inference_mode():
    saida_orig_reg = modelo_reg(X_test_reg)
    saida_orig_cls = modelo_cls(X_test_cls)

preds_orig_gpa = scaler_y_reg.inverse_transform(saida_orig_reg.numpy()).ravel()
reais_orig_gpa = scaler_y_reg.inverse_transform(y_test_reg.numpy().reshape(-1, 1)).ravel()
rmse_antes = float(np.sqrt(mean_squared_error(reais_orig_gpa, preds_orig_gpa)))

probs_orig_cls = torch.sigmoid(saida_orig_cls).numpy().ravel()
f1_antes = float(f1_score(y_test_cls.numpy().ravel().astype(int), (probs_orig_cls >= LIMIAR_CLS).astype(int)))

In [6]:
# 4. Salvamento dos checkpoints completos (.pt)
ckpt_reg = {
    "tarefa": "regressao",
    "state_dict": modelo_reg.state_dict(),
    "config": CFG_REG,
    "x_media": torch.tensor(scaler_X_reg.mean_, dtype=torch.float32),
    "x_desvio": torch.tensor(scaler_X_reg.scale_, dtype=torch.float32),
    "y_media": torch.tensor(scaler_y_reg.mean_, dtype=torch.float32),
    "y_desvio": torch.tensor(scaler_y_reg.scale_, dtype=torch.float32),
    "features": features_reg,
    "seed_selecionada": seed_reg,
}
torch.save(ckpt_reg, "../models/modelo_regressao.pt")

ckpt_cls = {
    "tarefa": "classificacao",
    "state_dict": modelo_cls.state_dict(),
    "config": CFG_CLS,
    "x_media": torch.tensor(scaler_X_cls.mean_, dtype=torch.float32),
    "x_desvio": torch.tensor(scaler_X_cls.scale_, dtype=torch.float32),
    "features": features_cls,
    "classes": ["Baixo_GPA (0)", "Alto_GPA (1)"],
    "limiar": LIMIAR_CLS,
    "seed_selecionada": seed_cls,
}
torch.save(ckpt_cls, "../models/modelo_classificacao.pt")

print(f"Modelo Regressão salvo (seed={seed_reg}) | RMSE Teste antes de salvar: {rmse_antes:.6f}")
print(f"Modelo Classificação salvo (seed={seed_cls}) | F1 Teste antes de salvar: {f1_antes:.6f}")

Modelo Regressão salvo (seed=100) | RMSE Teste antes de salvar: 1.750139
Modelo Classificação salvo (seed=42) | F1 Teste antes de salvar: 0.773723


In [7]:
# Remove os modelos da memória para simular carregamento em ambiente limpo
del modelo_reg, modelo_cls


def carregar_e_verificar(caminho_pt: str, X_teste: torch.Tensor, saida_esperada: torch.Tensor):
    # Portabilidade: map_location="cpu" garante funcionamento mesmo se treinado em GPU
    ckpt = torch.load(caminho_pt, map_location="cpu", weights_only=True)
    cfg = ckpt["config"]

    modelo_carregado = MLP(
        in_dim=cfg["in_dim"],
        out_dim=cfg["out_dim"],
        hidden=tuple(cfg["hidden"]),
        activation=cfg["activation"],
        dropout=cfg["dropout"],
    )
    modelo_carregado.load_state_dict(ckpt["state_dict"])
    modelo_carregado.eval()

    with torch.inference_mode():
        saida_nova = modelo_carregado(X_teste)

    identico = torch.allclose(saida_esperada, saida_nova, atol=1e-6)
    diff_max = float(torch.max(torch.abs(saida_esperada - saida_nova)).item())

    # Cálculo do tamanho em disco vs. tamanho teórico dos parâmetros (float32 = 4 bytes)
    n_params = modelo_carregado.contar_parametros()
    bytes_disco = os.path.getsize(caminho_pt)
    bytes_teoricos = n_params * 4
    overhead_bytes = bytes_disco - bytes_teoricos

    return modelo_carregado, ckpt, identico, diff_max, n_params, bytes_disco, bytes_teoricos, overhead_bytes

In [8]:
# 1. Verificação da Regressão
mod_rec_reg, ckpt_rec_reg, ok_reg, diff_reg, p_reg, bd_reg, bt_reg, ov_reg = carregar_e_verificar(
    "../models/modelo_regressao.pt", X_test_reg, saida_orig_reg
)
with torch.inference_mode():
    preds_depois_gpa = (
        mod_rec_reg(X_test_reg).numpy().ravel() * ckpt_rec_reg["y_desvio"].item() + ckpt_rec_reg["y_media"].item()
    )
rmse_depois = float(np.sqrt(mean_squared_error(reais_orig_gpa, preds_depois_gpa)))

# 2. Verificação da Classificação
mod_rec_cls, ckpt_rec_cls, ok_cls, diff_cls, p_cls, bd_cls, bt_cls, ov_cls = carregar_e_verificar(
    "../models/modelo_classificacao.pt", X_test_cls, saida_orig_cls
)
with torch.inference_mode():
    probs_depois_cls = torch.sigmoid(mod_rec_cls(X_test_cls)).numpy().ravel()
f1_depois = float(
    f1_score(y_test_cls.numpy().ravel().astype(int), (probs_depois_cls >= ckpt_rec_cls["limiar"]).astype(int))
)

In [10]:
# Relatório consolidado de fidelidade e armazenamento
df_persistencia = pd.DataFrame([
    {
        "Modelo": "modelo_regressao.pt",
        "Parâmetros": p_reg,
        "Teórico float32 (KB)": round(bt_reg / 1024, 2),
        "Disco Real (KB)": round(bd_reg / 1024, 2),
        "Overhead (Bytes)": ov_reg,
        "allclose": ok_reg,
        "Diff Máxima": f"{diff_reg:.2e}",
        "Métrica Antes": round(rmse_antes, 6),
        "Métrica Depois": round(rmse_depois, 6),
    },
    {
        "Modelo": "modelo_classificacao.pt",
        "Parâmetros": p_cls,
        "Teórico float32 (KB)": round(bt_cls / 1024, 2),
        "Disco Real (KB)": round(bd_cls / 1024, 2),
        "Overhead (Bytes)": ov_cls,
        "allclose": ok_cls,
        "Diff Máxima": f"{diff_cls:.2e}",
        "Métrica Antes": round(f1_antes, 6),
        "Métrica Depois": round(f1_depois, 6),
    },
])


df_persistencia.to_csv("../reports/tables/persistencia_modelos.csv", index=False)
print("Tabela salva em ../reports/tables/persistencia_modelos.csv")
display(df_persistencia)

Tabela salva em ../reports/tables/persistencia_modelos.csv


,Modelo,Parâmetros,Teórico float32 (KB),Disco Real (KB),Overhead (Bytes),allclose,Diff Máxima,Métrica Antes,Métrica Depois
0,modelo_regressao.pt,17409,68.0,72.46,4561,True,0.00e+00,1.750139,1.750139
1,modelo_classificacao.pt,17409,68.0,72.07,4159,True,0.00e+00,0.773723,0.773723
